### Databricks Structured Streaming

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import avg, count, round, col

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list("eventhub-scope1")

In [0]:
CONNECTION_STR = dbutils.secrets.get(
    scope="eventhub-scope1",
    key="databricksreader"
)

In [0]:
# Azure Event Hubs Connection String, Event Hub Namespace, name, and key

connectionConf = {
  'eventhubs.connectionString': sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(CONNECTION_STR),
  'eventhubs.name': 'banking-transactions'
}

In [0]:
# Reading stream to load data from Azure Event Hub into df Spark DataFrame
df = spark.readStream \
    .format("eventhubs") \
    .options(**connectionConf) \
    .load()

# Displaying stream 
display(df, checkpointLocation='/Volumes/bankingdata/bronze/bronze_log2')

In [0]:
# Writing the streaming data to the Delta table in append mode 
df.writeStream\
    .option("checkpointLocation", "/Volumes/bankingdata/bronze/bronze_log1")\
    .outputMode("append")\
    .format("delta")\
    .toTable("bankingdata.bronze.transactions")

In [0]:
%sql
select * from bankingdata.bronze.transactions

In [0]:
%sql
select count(*) from bankingdata.bronze.transactions